In [1]:
from google.colab import drive; drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import json

base = "/content/drive/MyDrive/sov"

paths = {
    "Core targets": "/content/ctx_core/targets.csv",
    "Core jobs": "/content/ctx_core/overlay/prompts/jobs_core.jsonl",
    "Multi-odd jobs": "/content/ctx_multi/overlay/prompts/jobs_multi_odd.jsonl",
    "Multi-even jobs": "/content/ctx_multi/overlay/prompts/jobs_multi_even.jsonl",
    "Dev jobs": "/content/drive/MyDrive/sov/ctx_dev/overlay/prompts/jobs_dev.jsonl",
    "Multi shuffled overlays": "/content/ctx_multi/overlay_shuffled",
}

for name, path in paths.items():
    print(f"{name}: {'FOUND' if os.path.exists(path) else 'MISSING'}")

print("\n--- Job counts ---")

job_files = {
    "multi_odd": paths["Multi-odd jobs"],
    "multi_even": paths["Multi-even jobs"],
    "core": paths["Core jobs"],
    "dev": paths["Dev jobs"],
}

for name, path in job_files.items():
    if os.path.isfile(path):
        with open(path, "r", encoding="utf-8") as f:
            count = sum(1 for line in f if line.strip())
        print(f"{name}: {count:,} jobs | estimated {count * 0.47 / 60:.1f} minutes")

Core targets: MISSING
Core jobs: MISSING
Multi-odd jobs: MISSING
Multi-even jobs: MISSING
Dev jobs: FOUND
Multi shuffled overlays: MISSING

--- Job counts ---
dev: 1,120 jobs | estimated 8.8 minutes


In [9]:
import os

base = "/content/drive/MyDrive/sov"

required_files = [
    "ctx_core_backup/core_set_context_images.zip",
    "ctx_core_backup/det/detections",
    "ctx_core_backup/prompts/jobs_core.jsonl",
    "ctx_core_backup/final",
    "ctx_core/overlay/prompts/jobs_core.jsonl",
    "ctx_core/overlay/final",
]

print("=== CORE BACKUP CHECK ===")

for item in required_files:
    path = os.path.join(base, item)
    exists = os.path.exists(path)

    if os.path.isfile(path):
        print(f"{item}: FOUND ({os.path.getsize(path) / 1024:.1f} KB)")
    elif os.path.isdir(path):
        print(f"{item}: FOUND (directory, {len(os.listdir(path))} items)")
    else:
        print(f"{item}: MISSING")

=== CORE BACKUP CHECK ===
ctx_core_backup/core_set_context_images.zip: FOUND (48404.7 KB)
ctx_core_backup/det/detections: FOUND (directory, 1473 items)
ctx_core_backup/prompts/jobs_core.jsonl: FOUND (3205.0 KB)
ctx_core_backup/final: FOUND (directory, 1473 items)
ctx_core/overlay/prompts/jobs_core.jsonl: MISSING
ctx_core/overlay/final: MISSING


In [7]:
scripts = [
    "05_match_and_jobs.py",
    "06_score_labels.py",
    "Sov_stage3_5_overlay.py",
]

for script in scripts:
    path = "/content/" + script
    print(script, "FOUND" if os.path.isfile(path) else "MISSING")

05_match_and_jobs.py FOUND
06_score_labels.py FOUND
Sov_stage3_5_overlay.py FOUND


In [10]:
import os
import shutil

backup = "/content/drive/MyDrive/sov/ctx_core_backup"
target = "/content/ctx_core"

folders = [
    ("det/detections", "det/detections"),
    ("final", "overlay/final"),
    ("prompts", "overlay/prompts"),
]

for src_rel, dst_rel in folders:
    src = os.path.join(backup, src_rel)
    dst = os.path.join(target, dst_rel)

    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)

    print("Restored:", dst_rel)

Restored: det/detections
Restored: overlay/final
Restored: overlay/prompts


In [11]:
import zipfile
from collections import Counter

zip_path = "/content/drive/MyDrive/sov/ctx_core_backup/core_set_context_images.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    names = z.namelist()

    print("ZIP integrity:", z.testzip())
    print("Total archive entries:", len(names))

    print("\nCSV files:")
    for name in names:
        if name.lower().endswith(".csv"):
            print(name)

    image_files = [
        n for n in names
        if n.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    print("\nTotal images:", len(image_files))
    print("First 15 image paths:")
    for name in image_files[:15]:
        print(name)

ZIP integrity: None
Total archive entries: 1474

CSV files:
targets.csv

Total images: 1473
First 15 image paths:
images/025395.jpg
images/009490.jpg
images/019570.jpg
images/008697.jpg
images/002112.jpg
images/011528.jpg
images/003133.jpg
images/005657.jpg
images/022654.jpg
images/009195.jpg
images/025003.jpg
images/006647.jpg
images/011028.jpg
images/025695.jpg
images/017695.jpg


In [12]:
import os

paths = [
    "/content/ctx_core/det/detections",
    "/content/ctx_core/overlay/final",
    "/content/ctx_core/overlay/prompts/jobs_core.jsonl",
]

for path in paths:
    print(
        os.path.basename(path),
        "FOUND" if os.path.exists(path) else "MISSING"
    )

detections FOUND
final FOUND
jobs_core.jsonl FOUND


In [13]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/sov/ctx_core_backup/core_set_context_images.zip"
extract_dir = "/content/ctx_core"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

print("Images:", len(os.listdir("/content/ctx_core/images")))
print("Targets CSV:", os.path.exists("/content/ctx_core/targets.csv"))

Images: 1473
Targets CSV: True


In [14]:
!python Sov_stage3_5_overlay.py \
  --input /content/ctx_core/images \
  --detections /content/ctx_core/det/detections \
  --output /content/ctx_core/overlay \
  --eps 0.4

05:44:56 INFO Found 1473 detection file(s). Settings: {'eps': 0.4, 'style_version': 1, 'numbering': 'left_to_right'}
05:44:56 INFO [1/1473] 000000.jpg: 1 detected -> 1 final (0 dropped) in 0.3s
05:44:56 INFO [2/1473] 000005.jpg: 1 detected -> 1 final (0 dropped) in 0.2s
05:44:56 INFO [3/1473] 000007.jpg: 1 detected -> 1 final (0 dropped) in 0.1s
05:44:56 INFO [4/1473] 000009.jpg: 1 detected -> 1 final (0 dropped) in 0.1s
05:44:56 INFO [5/1473] 000014.jpg: 1 detected -> 1 final (0 dropped) in 0.2s
05:44:57 INFO [6/1473] 000015.jpg: 1 detected -> 1 final (0 dropped) in 0.2s
05:44:57 INFO [7/1473] 000020.jpg: 1 detected -> 1 final (0 dropped) in 0.2s
05:44:57 INFO [8/1473] 000027.jpg: 2 detected -> 2 final (0 dropped) in 0.2s
05:44:57 INFO [9/1473] 000043.jpg: 1 detected -> 1 final (0 dropped) in 0.1s
05:44:57 INFO [10/1473] 000048.jpg: 1 detected -> 1 final (0 dropped) in 0.2s
05:44:58 INFO [11/1473] 000051.jpg: 1 detected -> 1 final (0 dropped) in 0.2s
05:44:58 INFO [12/1473] 000059.jpg

In [15]:
import os

base = "/content/ctx_core/overlay/overlays"

print("Image folders:", len(os.listdir(base)))

sample = "025395.jpg"

for condition in ["plain", "box", "box_number", "sov"]:
    path = f"{base}/{sample}/{condition}.png"
    print(condition, os.path.exists(path))

Image folders: 1473
plain True
box True
box_number True
sov True


In [17]:
!pip install -q -U transformers accelerate bitsandbytes

In [18]:
!python /content/06_score_labels.py \
  --model qwen25vl \
  --overlay_root /content/ctx_core/overlay \
  --jobs /content/ctx_core/overlay/prompts/jobs_core.jsonl \
  --out_dir /content/drive/MyDrive/sov/results_core_qwen

6428 job(s) | model=qwen25vl | out_dir=/content/drive/MyDrive/sov/results_core_qwen
model.safetensors.index.json: 100% 57.6k/57.6k [00:00<00:00, 107MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0% 0/5 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/1.09G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/4.99G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/16.6G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/16.6G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   8% 1.39G/16.6G [00:15<03:01, 83.9MB/s,  150MB/s  ]
Reconstructing (incomplete total...):  17% 2.81G/16.6G [00:23<01:55, 119MB/s,  143MB/s  ]
Reconstructing (incomplete total...):  20% 3.30G/16.6G [00:32<03:31, 62.7MB/s, 97.9MB/s  ]
Reconstructing (incomplete total...):  26% 4.25G/16.6G [00:47<03:41, 55.6MB/s, 97.3MB/s  ]

Fetching 5 files:  20% 1/5 [00:48<03:13, 48.38s/it]
Reconstructing (i

In [19]:
import os

base = "/content/drive/MyDrive/sov"

print("=== MULTI-FACE FILE DISCOVERY ===")

for root, dirs, files in os.walk(base):
    # Avoid listing unrelated result files
    relevant = [
        f for f in files
        if (
            "multi_odd" in f.lower()
            or "multi_even" in f.lower()
            or f == "targets_view.csv"
            or f == "core_set_context_images.zip"
        )
    ]

    for f in relevant:
        path = os.path.join(root, f)
        print(path, "|", round(os.path.getsize(path) / (1024**2), 2), "MB")

=== MULTI-FACE FILE DISCOVERY ===
/content/drive/MyDrive/sov/ctx_core_backup/core_set_context_images.zip | 47.27 MB
/content/drive/MyDrive/sov/results_multi_even/results_llava15_jobs_multi_even.jsonl | 4.44 MB
/content/drive/MyDrive/sov/multiface/jobs_multi_even.jsonl | 3.81 MB
/content/drive/MyDrive/sov/multiface/targets_view.csv | 0.78 MB
/content/drive/MyDrive/sov/qwen_smoke_test/results_qwen25vl_jobs_multi_even.jsonl | 0.01 MB
/content/drive/MyDrive/sov/results_multi_even_qwen/results_qwen25vl_jobs_multi_even.jsonl | 4.73 MB


In [23]:
import zipfile

zip_path = "/content/drive/MyDrive/sov_teammate/multi_group_images.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    names = z.namelist()

    print("ZIP integrity:", z.testzip())
    print("Total entries:", len(names))

    images = [
        n for n in names
        if n.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    csvs = [n for n in names if n.lower().endswith(".csv")]

    print("Image count:", len(images))
    print("CSV files:", csvs)
    print("First 10 image paths:")

    for name in images[:10]:
        print(name)

ZIP integrity: None
Total entries: 1251
Image count: 1250
CSV files: ['targets_view.csv']
First 10 image paths:
images/m00000.jpg
images/m00001.jpg
images/m00002.jpg
images/m00003.jpg
images/m00004.jpg
images/m00005.jpg
images/m00006.jpg
images/m00007.jpg
images/m00008.jpg
images/m00009.jpg


In [25]:
import zipfile
import os
import hashlib

zip_path = "/content/drive/MyDrive/sov_teammate/multi_group_images.zip"
target_csv = "/content/ctx_multi/targets_view.csv"

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

with zipfile.ZipFile(zip_path, "r") as z:
    image_members = [
        n for n in z.namelist()
        if n.startswith("images/") and n.lower().endswith(".jpg")
    ]

    # Check whether the archive CSV matches the restored CSV
    archive_csv = z.read("targets_view.csv")
    restored_csv = open(target_csv, "rb").read()

    print("Targets CSV matches archive:", archive_csv == restored_csv)

    z.extractall(
        "/content/ctx_multi",
        members=image_members
    )

print("Extracted images:", len(os.listdir("/content/ctx_multi/images")))

Targets CSV matches archive: True
Extracted images: 1250


In [26]:
%cd /content

!python sov_stage3_5_overlay_scaled.py \
  --input /content/ctx_multi/images \
  --detections /content/ctx_multi/det/detections \
  --output /content/ctx_multi/overlay \
  --eps 0.4

/content
06:55:46 INFO Found 1250 detection file(s). Settings: {'eps': 0.4, 'style_version': 2, 'numbering': 'left_to_right', 'spec': 'scaled', 'line_frac': 0.03, 'dot_frac': 0.025, 'tag_frac': 0.28, 'shuffle_numbers': None}
06:55:46 INFO [1/1250] m00000.jpg: 9 detected -> 9 final (0 dropped) in 0.4s
06:55:47 INFO [2/1250] m00001.jpg: 2 detected -> 2 final (0 dropped) in 0.2s
06:55:47 INFO [3/1250] m00002.jpg: 9 detected -> 9 final (0 dropped) in 0.2s
06:55:47 INFO [4/1250] m00003.jpg: 2 detected -> 2 final (0 dropped) in 0.1s
06:55:47 INFO [5/1250] m00004.jpg: 3 detected -> 3 final (0 dropped) in 0.1s
06:55:47 INFO [6/1250] m00005.jpg: 2 detected -> 2 final (0 dropped) in 0.1s
06:55:47 INFO [7/1250] m00006.jpg: 2 detected -> 2 final (0 dropped) in 0.1s
06:55:48 INFO [8/1250] m00007.jpg: 6 detected -> 6 final (0 dropped) in 0.2s
06:55:48 INFO [9/1250] m00008.jpg: 6 detected -> 6 final (0 dropped) in 0.1s
06:55:48 INFO [10/1250] m00009.jpg: 2 detected -> 2 final (0 dropped) in 0.2s
06:5

In [27]:
!python 11_match_multiface_jobs.py \
  --targets /content/ctx_multi/targets_view.csv \
  --overlay_root /content/ctx_multi/overlay \
  --name multi_odd \
  --chunk_mod 2 \
  --chunk_rem 1

target faces: 2818 | matched: 2816 (99.9%) | jobs written: 6400 (chunk_mod=2, chunk_rem=1)
not matched, by reason: {'no_unique_match': 2}
IoU of matched faces, quantiles 10/50/90: [0.58, 0.69, 0.77]
images where the detector found a different number of faces than we have targets: 1206 faces of 2816

match rate by race_group:
race_group
Asian              99.8
Black             100.0
Caucasian-like     99.9

match rate by gender15:
gender15
Female    99.9
Male      99.9

match rate by mixed:
mixed
False    99.9
True     99.9

match rate by face size in the view:
size_tier
small      99.9
medium    100.0
large      99.9

jobs file: /content/ctx_multi/overlay/prompts/jobs_multi_odd.jsonl


In [28]:
!python 06_score_labels.py \
  --model qwen25vl \
  --overlay_root /content/ctx_multi/overlay \
  --jobs /content/ctx_multi/overlay/prompts/jobs_multi_odd.jsonl \
  --out_dir /content/drive/MyDrive/sov/results_multi_odd_qwen

6400 job(s) | model=qwen25vl | out_dir=/content/drive/MyDrive/sov/results_multi_odd_qwen
Loading weights: 100% 729/729 [01:07<00:00, 10.79it/s]
0 already done, 6400 to run.
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(
[25/6400] 0.49s/job (last 50) | ~0.87 h left
[50/6400] 0.44s/job (last 50) | ~0.77 h left
[75/6400] 0.38s/job (last 50) | ~0.67 h left
[100/6400] 0.39s/job (last 50) | ~0.68 h left
[125/6400] 0.40s/job (last 50) | ~0.69 h left
[150/6400] 0.40s/job (last 50) | ~0.70 h left
[175/6400] 0.41s/job (last 50) | ~0.71 h left
[200/6400] 0.42s/job (last 50) | ~0.72 h left
[225/6400] 0.42s/job (last 50) | ~0.73 h left
[250/6400] 0.43s/job (last 50) | ~0.74 h left
[275/6400] 0.44s/job (last 50) | ~0.75 h left
[300/6400] 0.45s/job (last 50) | ~0.76 h left
[325/6400] 0.46s/job (last 50) | ~0.77 h left
[350/6400] 0.46s/job

In [29]:
!python /content/sov_stage3_5_overlay_scaled.py \
  --input /content/ctx_multi/images \
  --detections /content/ctx_multi/det/detections \
  --output /content/ctx_multi/overlay_shuffled \
  --shuffle_numbers 7

08:44:53 INFO Found 1250 detection file(s). Settings: {'eps': 0.4, 'style_version': 2, 'numbering': 'left_to_right', 'spec': 'scaled', 'line_frac': 0.03, 'dot_frac': 0.025, 'tag_frac': 0.28, 'shuffle_numbers': 7}
08:44:53 INFO [1/1250] m00000.jpg: 9 detected -> 9 final (0 dropped) in 0.4s
08:44:53 INFO [2/1250] m00001.jpg: 2 detected -> 2 final (0 dropped) in 0.2s
08:44:54 INFO [3/1250] m00002.jpg: 9 detected -> 9 final (0 dropped) in 0.2s
08:44:54 INFO [4/1250] m00003.jpg: 2 detected -> 2 final (0 dropped) in 0.1s
08:44:54 INFO [5/1250] m00004.jpg: 3 detected -> 3 final (0 dropped) in 0.1s
08:44:54 INFO [6/1250] m00005.jpg: 2 detected -> 2 final (0 dropped) in 0.1s
08:44:54 INFO [7/1250] m00006.jpg: 2 detected -> 2 final (0 dropped) in 0.1s
08:44:54 INFO [8/1250] m00007.jpg: 6 detected -> 6 final (0 dropped) in 0.2s
08:44:54 INFO [9/1250] m00008.jpg: 6 detected -> 6 final (0 dropped) in 0.1s
08:44:55 INFO [10/1250] m00009.jpg: 2 detected -> 2 final (0 dropped) in 0.2s
08:44:55 INFO [1

In [30]:
import os

root = "/content/ctx_multi/overlay_shuffled"

print("Overlay root:", os.path.exists(root))
print("Final metadata:", len(os.listdir(root + "/final")))

sample = os.listdir(root + "/final")[0]
key = sample.removesuffix(".json")

for condition in ["box_number", "sov"]:
    path = f"{root}/overlays/{key}/{condition}.png"
    print(condition, os.path.exists(path))

Overlay root: True
Final metadata: 1250
box_number True
sov True


In [1]:
!python /content/12_tag_probe.py \
  --overlay_root /content/ctx_multi/overlay_shuffled \
  --model qwen25vl \
  --out_dir /content/drive/MyDrive/sov/probe \
  --n_images 150

python3: can't open file '/content/12_tag_probe.py': [Errno 2] No such file or directory
